# PitchIQ — Match Outcome Model

**Poisson regression and the Dixon–Coles extension**

The exploratory analysis established three facts that determine everything below:

- Goal counts are statistically consistent with a Poisson distribution (p = 0.31 home, p = 0.46 away)
- Home advantage is real and worth roughly +0.25 goals per match
- Always predicting a home win yields ~43.5% accuracy — the floor any model must clear

This notebook builds two models on that foundation, evaluates them honestly against a
held-out season, and produces probabilistic forecasts for upcoming fixtures.

**Approach**

| Model | Description |
|---|---|
| Baseline | Historical outcome frequencies — no learning at all |
| Poisson GLM | Team attack/defense effects + home advantage, fit by maximum likelihood |
| Dixon–Coles | Poisson with a low-score dependence correction and time-weighted fitting |

**Evaluation philosophy.** Accuracy alone is a poor metric for football. A model that says
"55% home win" and one that says "95% home win" score identically when the home team wins, yet
they are very different models. The primary metrics here are **log-loss** and **ranked
probability score**, which reward well-calibrated probabilities rather than confident guesses.


## 1. Setup

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import accuracy_score, log_loss

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "predictions"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Load data and split chronologically

In [ ]:
files = sorted(DATA_DIR.glob("pl_matches_*.csv"))
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df["utc_date"] = pd.to_datetime(df["utc_date"])
df = df.sort_values("utc_date").reset_index(drop=True)

played = df[df["status"] == "FINISHED"].copy()
upcoming = df[df["status"] != "FINISHED"].copy()

seasons = sorted(played["season"].unique())
print(f"Seasons: {seasons}")
print(f"Played:  {len(played):,}")
print(f"Upcoming: {len(upcoming):,}")

### Why the split is chronological

A random train/test split would let the model learn from matches that happened *after* the
ones it is asked to predict. In football that is a serious leak: mid-season transfers, manager
changes and injuries all mean future matches carry information about the present.

The split below mirrors how the model would actually be used — fit on the past, forecast the
future.

In [ ]:
# Reserve the most recent complete season for evaluation.
complete_seasons = [s for s in seasons if (played["season"] == s).sum() >= 300]
test_season = complete_seasons[-1]
train_seasons = complete_seasons[:-1]

train = played[played["season"].isin(train_seasons)].copy()
test = played[played["season"] == test_season].copy()

print(f"Train seasons: {train_seasons}  ->  {len(train):,} matches")
print(f"Test season:   {test_season}      ->  {len(test):,} matches")
print(f"\nTrain window: {train['utc_date'].min().date()} to {train['utc_date'].max().date()}")
print(f"Test window:  {test['utc_date'].min().date()} to {test['utc_date'].max().date()}")

assert train["utc_date"].max() < test["utc_date"].min(), "Temporal leak detected"
print("\nNo temporal overlap between train and test.")

## 3. Shared machinery

Both models produce two expected goal rates per match (λ for home, μ for away). Converting
those into win/draw/loss probabilities requires building the full matrix of plausible
scorelines and summing the regions that correspond to each outcome.

In [ ]:
MAX_GOALS = 10  # P(more than 10 goals for one side) is negligible


def score_matrix(lam_home, lam_away, rho=0.0, max_goals=MAX_GOALS):
    """Joint probability of every scoreline up to max_goals.

    rho applies the Dixon-Coles correction for low-scoring matches, where the
    independence assumption of a plain double-Poisson is known to break down.
    """
    p_home = stats.poisson.pmf(np.arange(max_goals + 1), lam_home)
    p_away = stats.poisson.pmf(np.arange(max_goals + 1), lam_away)
    matrix = np.outer(p_home, p_away)

    if rho != 0.0:
        matrix[0, 0] *= 1 - lam_home * lam_away * rho
        matrix[0, 1] *= 1 + lam_home * rho
        matrix[1, 0] *= 1 + lam_away * rho
        matrix[1, 1] *= 1 - rho
        matrix = np.clip(matrix, 1e-12, None)
        matrix /= matrix.sum()

    return matrix


def outcome_probs(lam_home, lam_away, rho=0.0):
    """Return (P_home_win, P_draw, P_away_win)."""
    m = score_matrix(lam_home, lam_away, rho)
    return np.tril(m, -1).sum(), np.trace(m), np.triu(m, 1).sum()


def ranked_probability_score(probs, y_true):
    """RPS — the standard metric for ordered football forecasts.

    Unlike log-loss it penalises *how far* a prediction was from the truth:
    predicting a home win when the away team won is worse than predicting a draw.
    """
    onehot = np.zeros_like(probs)
    onehot[np.arange(len(y_true)), y_true] = 1
    cum_p, cum_o = np.cumsum(probs, axis=1), np.cumsum(onehot, axis=1)
    return np.mean(np.sum((cum_p - cum_o) ** 2, axis=1) / (probs.shape[1] - 1))


OUTCOME_MAP = {"H": 0, "D": 1, "A": 2}
y_test = test["result"].map(OUTCOME_MAP).values

## 4. Baseline

The number to beat. No features, no learning — just the historical frequency of each outcome.

In [ ]:
base_rates = (
    train["result"].value_counts(normalize=True).reindex(["H", "D", "A"]).values
)
baseline_probs = np.tile(base_rates, (len(test), 1))

results = {}
results["Baseline (frequencies)"] = {
    "accuracy": accuracy_score(y_test, np.full(len(test), 0)),
    "log_loss": log_loss(y_test, baseline_probs, labels=[0, 1, 2]),
    "rps": ranked_probability_score(baseline_probs, y_test),
}

print(f"Historical rates — H: {base_rates[0]:.1%}  D: {base_rates[1]:.1%}  A: {base_rates[2]:.1%}")
print(f"\nBaseline accuracy: {results['Baseline (frequencies)']['accuracy']:.1%}")
print(f"Baseline log-loss: {results['Baseline (frequencies)']['log_loss']:.4f}")

## 5. Model 1 — Poisson GLM

Each match contributes two observations: goals scored by the home side, and goals scored by
the away side. A generalised linear model with a log link then estimates, for every team, an
**attack** coefficient (how many goals it tends to score) and a **defense** coefficient (how
many it tends to concede), plus a single shared home-advantage term.

Formally, for team *i* facing team *j*:

$$\log(\lambda_{ij}) = \mu + \alpha_i + \beta_j + \gamma \cdot \mathbb{1}[\text{home}]$$

In [ ]:
def to_long_format(data: pd.DataFrame) -> pd.DataFrame:
    """Reshape one row per match into two rows: one per scoring team."""
    home = pd.DataFrame({
        "goals": data["home_goals"].astype(int),
        "team": data["home_team"],
        "opponent": data["away_team"],
        "home": 1,
    })
    away = pd.DataFrame({
        "goals": data["away_goals"].astype(int),
        "team": data["away_team"],
        "opponent": data["home_team"],
        "home": 0,
    })
    return pd.concat([home, away], ignore_index=True)


long_train = to_long_format(train)
print(f"{len(train):,} matches  ->  {len(long_train):,} team-match observations")
long_train.head()

In [ ]:
glm = smf.glm(
    formula="goals ~ home + C(team) + C(opponent)",
    data=long_train,
    family=sm.families.Poisson(),
).fit()

print(f"Parameters estimated: {len(glm.params)}")
print(f"Home advantage (log scale): {glm.params['home']:.4f}")
print(f"Multiplicative effect: {np.exp(glm.params['home']):.3f}x more goals at home")
print(f"\nDeviance: {glm.deviance:.1f}  |  AIC: {glm.aic:.1f}")

In [ ]:
train_teams = set(long_train["team"])
league_home_avg = train["home_goals"].mean()
league_away_avg = train["away_goals"].mean()


def glm_predict(home_team: str, away_team: str) -> tuple[float, float]:
    """Expected goals for both sides. Falls back to league averages for
    promoted teams the model has never seen."""
    if home_team not in train_teams or away_team not in train_teams:
        return league_home_avg, league_away_avg

    row_home = pd.DataFrame([{"team": home_team, "opponent": away_team, "home": 1}])
    row_away = pd.DataFrame([{"team": away_team, "opponent": home_team, "home": 0}])
    return float(glm.predict(row_home).iloc[0]), float(glm.predict(row_away).iloc[0])


glm_probs = np.array([
    outcome_probs(*glm_predict(r.home_team, r.away_team)) for r in test.itertuples()
])

results["Poisson GLM"] = {
    "accuracy": accuracy_score(y_test, glm_probs.argmax(axis=1)),
    "log_loss": log_loss(y_test, glm_probs, labels=[0, 1, 2]),
    "rps": ranked_probability_score(glm_probs, y_test),
}

for k, v in results["Poisson GLM"].items():
    print(f"{k:>10}: {v:.4f}")

## 6. Model 2 — Dixon–Coles

The GLM assumes home and away goals are independent. Dixon & Coles (1997) showed this fails
for low scores: 0–0 and 1–1 occur more often than independence predicts, while 1–0 and 0–1
occur less often. Their fix multiplies those four cells by a correction factor τ governed by a
single parameter ρ.

Two further refinements:

- **Time decay.** Recent matches carry more information than old ones. Each match is weighted
  by exp(−ξ · days_ago), so form fades gradually instead of counting equally forever.
- **Sum-to-zero constraint** on attack parameters, needed for identifiability — otherwise
  adding a constant to every attack and subtracting it from every defense gives an identical
  fit.

In [ ]:
teams = sorted(set(train["home_team"]) | set(train["away_team"]))
team_index = {t: i for i, t in enumerate(teams)}
n_teams = len(teams)

goals_home = train["home_goals"].astype(int).values
goals_away = train["away_goals"].astype(int).values
idx_home = train["home_team"].map(team_index).values
idx_away = train["away_team"].map(team_index).values

reference_date = train["utc_date"].max()
days_ago = (reference_date - train["utc_date"]).dt.days.values

print(f"Teams in training window: {n_teams}")
print(f"Free parameters: {2 * n_teams + 1}  (attack, defense, home advantage, rho)")

In [ ]:
def dixon_coles_tau(x, y, lam, mu, rho):
    """Low-score correction factor. Returns 1 for every scoreline above 1-1."""
    tau = np.ones_like(lam, dtype=float)
    tau[(x == 0) & (y == 0)] = (1 - lam * mu * rho)[(x == 0) & (y == 0)]
    tau[(x == 0) & (y == 1)] = (1 + lam * rho)[(x == 0) & (y == 1)]
    tau[(x == 1) & (y == 0)] = (1 + mu * rho)[(x == 1) & (y == 0)]
    tau[(x == 1) & (y == 1)] = 1 - rho
    return np.clip(tau, 1e-10, None)


def unpack(params):
    """Rebuild the constrained parameter vector."""
    attack_free = params[:n_teams - 1]
    attack = np.concatenate([attack_free, [-attack_free.sum()]])  # sums to zero
    defense = params[n_teams - 1:2 * n_teams - 1]
    return attack, defense, params[2 * n_teams - 1], params[2 * n_teams]


def negative_log_likelihood(params, xi=0.0):
    attack, defense, home_adv, rho = unpack(params)

    lam = np.exp(attack[idx_home] - defense[idx_away] + home_adv)
    mu = np.exp(attack[idx_away] - defense[idx_home])

    weights = np.exp(-xi * days_ago) if xi > 0 else 1.0

    ll = (
        stats.poisson.logpmf(goals_home, lam)
        + stats.poisson.logpmf(goals_away, mu)
        + np.log(dixon_coles_tau(goals_home, goals_away, lam, mu, rho))
    )
    return -np.sum(weights * ll)


x0 = np.concatenate([np.zeros(n_teams - 1), np.zeros(n_teams), [0.25], [-0.05]])
bounds = ([(-3, 3)] * (n_teams - 1) + [(-3, 3)] * n_teams + [(-1, 1), (-0.2, 0.2)])

In [ ]:
# Tune the time-decay rate by validating on the held-out season.
# xi = 0 means every match counts equally; higher values discount the past faster.
decay_results = []

for xi in [0.0, 0.0005, 0.001, 0.002, 0.003, 0.005]:
    fit = minimize(negative_log_likelihood, x0, args=(xi,),
                   method="L-BFGS-B", bounds=bounds, options={"maxiter": 2000})
    attack, defense, home_adv, rho = unpack(fit.x)

    probs = []
    for r in test.itertuples():
        i, j = team_index.get(r.home_team), team_index.get(r.away_team)
        a_h = attack[i] if i is not None else attack.mean()
        d_h = defense[i] if i is not None else defense.mean()
        a_a = attack[j] if j is not None else attack.mean()
        d_a = defense[j] if j is not None else defense.mean()
        probs.append(outcome_probs(np.exp(a_h - d_a + home_adv), np.exp(a_a - d_h), rho))

    probs = np.array(probs)
    decay_results.append({
        "xi": xi,
        "half_life_days": np.log(2) / xi if xi > 0 else np.inf,
        "log_loss": log_loss(y_test, probs, labels=[0, 1, 2]),
        "rps": ranked_probability_score(probs, y_test),
        "accuracy": accuracy_score(y_test, probs.argmax(axis=1)),
    })

decay_df = pd.DataFrame(decay_results)
decay_df.round(4)

In [ ]:
best_xi = decay_df.loc[decay_df["rps"].idxmin(), "xi"]
print(f"Selected decay rate: xi = {best_xi}")
if best_xi > 0:
    print(f"Half-life: {np.log(2) / best_xi:.0f} days")

dc_fit = minimize(negative_log_likelihood, x0, args=(best_xi,),
                  method="L-BFGS-B", bounds=bounds, options={"maxiter": 3000})
attack, defense, home_adv, rho = unpack(dc_fit.x)

print(f"\nConverged: {dc_fit.success}")
print(f"Home advantage: {home_adv:.4f}  ({np.exp(home_adv):.3f}x)")
print(f"Rho (low-score correction): {rho:.4f}")

In [ ]:
def dc_predict(home_team: str, away_team: str) -> tuple[float, float]:
    """Expected goals under Dixon-Coles. Unseen teams receive
    replacement-level parameters rather than being dropped."""
    i, j = team_index.get(home_team), team_index.get(away_team)
    a_h = attack[i] if i is not None else attack.mean()
    d_h = defense[i] if i is not None else defense.mean()
    a_a = attack[j] if j is not None else attack.mean()
    d_a = defense[j] if j is not None else defense.mean()
    return np.exp(a_h - d_a + home_adv), np.exp(a_a - d_h)


dc_probs = np.array([
    outcome_probs(*dc_predict(r.home_team, r.away_team), rho)
    for r in test.itertuples()
])

results["Dixon-Coles"] = {
    "accuracy": accuracy_score(y_test, dc_probs.argmax(axis=1)),
    "log_loss": log_loss(y_test, dc_probs, labels=[0, 1, 2]),
    "rps": ranked_probability_score(dc_probs, y_test),
}

for k, v in results["Dixon-Coles"].items():
    print(f"{k:>10}: {v:.4f}")

## 7. Model comparison

In [ ]:
comparison = pd.DataFrame(results).T
comparison["vs baseline (log-loss)"] = (
    (results["Baseline (frequencies)"]["log_loss"] - comparison["log_loss"])
    / results["Baseline (frequencies)"]["log_loss"] * 100
).round(1)
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = [("accuracy", "Accuracy (higher is better)", True),
           ("log_loss", "Log-loss (lower is better)", False),
           ("rps", "Ranked probability score (lower is better)", False)]
palette = ["#8d99ae", "#457b9d", "#2a9d8f"]

for ax, (metric, title, higher_better) in zip(axes, metrics):
    values = comparison[metric].values
    bars = ax.bar(range(len(values)), values, color=palette,
                  edgecolor="white", linewidth=1.5)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontweight="bold", fontsize=9)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(["Baseline", "GLM", "Dixon-Coles"], fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, max(values) * 1.25)

plt.tight_layout()
plt.show()

### Calibration

The decisive test for a probabilistic model. If the model says "60% chance of a home win"
across many matches, the home team should win about 60% of the time. A model can be accurate
and badly calibrated — and for anything involving decisions under uncertainty, calibration
matters more.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5))
bins = np.linspace(0, 1, 11)

for probs, label, color in [(dc_probs, "Dixon-Coles", "#2a9d8f"),
                            (glm_probs, "Poisson GLM", "#457b9d")]:
    flat_p = probs.ravel()
    flat_y = np.zeros_like(probs)
    flat_y[np.arange(len(y_test)), y_test] = 1
    flat_y = flat_y.ravel()

    idx = np.digitize(flat_p, bins) - 1
    xs, ys, ns = [], [], []
    for b in range(len(bins) - 1):
        mask = idx == b
        if mask.sum() >= 10:
            xs.append(flat_p[mask].mean())
            ys.append(flat_y[mask].mean())
            ns.append(mask.sum())

    ax.plot(xs, ys, "o-", color=color, linewidth=2, markersize=7, label=label)

ax.plot([0, 1], [0, 1], "--", color="#8d99ae", linewidth=1.5, label="Perfect calibration")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Observed frequency")
ax.set_title("Calibration curve")
ax.legend(frameon=False, loc="upper left")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 8. What the model learned

The fitted attack and defense ratings — the model's view of the league.

In [ ]:
ratings = pd.DataFrame({
    "team": teams,
    "attack": attack,
    "defense": defense,
}).assign(
    overall=lambda d: d["attack"] + d["defense"]
).sort_values("overall", ascending=False).reset_index(drop=True)

ratings.round(3).head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(ratings["attack"], ratings["defense"], s=100, c="#264653",
           alpha=0.75, edgecolor="white", linewidth=1.5, zorder=3)

for _, row in ratings.iterrows():
    ax.annotate(row["team"].replace(" FC", "").replace(" AFC", ""),
                (row["attack"], row["defense"]), fontsize=8,
                xytext=(6, 3), textcoords="offset points", alpha=0.85)

ax.axhline(0, color="#8d99ae", ls="--", lw=1)
ax.axvline(0, color="#8d99ae", ls="--", lw=1)
ax.set_xlabel("Attack rating  →  scores more")
ax.set_ylabel("Defense rating  →  concedes less")
ax.set_title("Estimated team strength (Dixon-Coles parameters)")
plt.tight_layout()
plt.show()

## 9. Refit on all available data and forecast

The evaluation above used only the training seasons so the test season stayed genuinely
unseen. For live forecasting there is no reason to withhold data — the model is refit on every
completed match before predicting fixtures that have not been played.

In [ ]:
# Rebuild the index over the full history, including the current season.
full = played.copy()
teams = sorted(set(full["home_team"]) | set(full["away_team"]))
team_index = {t: i for i, t in enumerate(teams)}
n_teams = len(teams)

goals_home = full["home_goals"].astype(int).values
goals_away = full["away_goals"].astype(int).values
idx_home = full["home_team"].map(team_index).values
idx_away = full["away_team"].map(team_index).values
reference_date = full["utc_date"].max()
days_ago = (reference_date - full["utc_date"]).dt.days.values

x0 = np.concatenate([np.zeros(n_teams - 1), np.zeros(n_teams), [0.25], [-0.05]])
bounds = ([(-3, 3)] * (n_teams - 1) + [(-3, 3)] * n_teams + [(-1, 1), (-0.2, 0.2)])

final_fit = minimize(negative_log_likelihood, x0, args=(best_xi,),
                     method="L-BFGS-B", bounds=bounds, options={"maxiter": 3000})
attack, defense, home_adv, rho = unpack(final_fit.x)

print(f"Refit on {len(full):,} matches across {n_teams} teams.")
print(f"Converged: {final_fit.success}")

In [ ]:
forecasts = []

for r in upcoming.itertuples():
    lam, mu = dc_predict(r.home_team, r.away_team)
    p_home, p_draw, p_away = outcome_probs(lam, mu, rho)
    matrix = score_matrix(lam, mu, rho)
    likely = np.unravel_index(matrix.argmax(), matrix.shape)

    forecasts.append({
        "match_id": r.match_id,
        "date": r.utc_date.date(),
        "matchday": r.matchday,
        "home_team": r.home_team,
        "away_team": r.away_team,
        "xg_home": round(lam, 2),
        "xg_away": round(mu, 2),
        "p_home_win": round(p_home, 4),
        "p_draw": round(p_draw, 4),
        "p_away_win": round(p_away, 4),
        "most_likely_score": f"{likely[0]}-{likely[1]}",
        "prediction": ["H", "D", "A"][int(np.argmax([p_home, p_draw, p_away]))],
        "confidence": round(max(p_home, p_draw, p_away), 4),
    })

forecast_df = pd.DataFrame(forecasts).sort_values("date").reset_index(drop=True)
print(f"Generated {len(forecast_df):,} forecasts.")
forecast_df.head(10)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
forecast_path = OUTPUT_DIR / "upcoming_forecasts.csv"
forecast_df.to_csv(forecast_path, index=False)

ratings_final = pd.DataFrame({
    "team": teams, "attack": attack, "defense": defense
}).assign(overall=lambda d: d["attack"] + d["defense"]).sort_values(
    "overall", ascending=False
).reset_index(drop=True)
ratings_final.round(4).to_csv(OUTPUT_DIR / "team_ratings.csv", index=False)

print(f"Saved forecasts to {forecast_path.relative_to(PROJECT_ROOT)}")
print(f"Saved ratings to   {(OUTPUT_DIR / 'team_ratings.csv').relative_to(PROJECT_ROOT)}")

### Next fixtures

In [ ]:
next_up = forecast_df.head(10).copy()

fig, ax = plt.subplots(figsize=(11, 5.5))
y = np.arange(len(next_up))
labels = [f"{r.home_team.replace(' FC','')} v {r.away_team.replace(' FC','')}"
          for r in next_up.itertuples()]

ax.barh(y, next_up["p_home_win"], color="#2a9d8f", label="Home win", edgecolor="white")
ax.barh(y, next_up["p_draw"], left=next_up["p_home_win"], color="#e9c46a",
        label="Draw", edgecolor="white")
ax.barh(y, next_up["p_away_win"],
        left=next_up["p_home_win"] + next_up["p_draw"], color="#e76f51",
        label="Away win", edgecolor="white")

ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("Probability")
ax.set_title("Forecasts for the next fixtures")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

## 10. Findings and limitations

In [ ]:
best = comparison["rps"].idxmin()
improvement = ((results["Baseline (frequencies)"]["rps"] - results[best]["rps"])
               / results["Baseline (frequencies)"]["rps"] * 100)

print("SUMMARY")
print("=" * 60)
print(f"Best model:              {best}")
print(f"Test season:             {test_season}  ({len(test)} matches)")
print(f"Accuracy:                {results[best]['accuracy']:.1%}  "
      f"(baseline {results['Baseline (frequencies)']['accuracy']:.1%})")
print(f"Log-loss:                {results[best]['log_loss']:.4f}  "
      f"(baseline {results['Baseline (frequencies)']['log_loss']:.4f})")
print(f"RPS improvement:         {improvement:.1f}% over baseline")
print(f"Home advantage:          {np.exp(home_adv):.3f}x goal rate")
print(f"Forecasts generated:     {len(forecast_df):,}")
print("=" * 60)

### What worked

- **Dixon–Coles outperforms the plain GLM** on log-loss and RPS, confirming that the
  low-score correction captures a real dependency rather than adding noise.
- **Both models beat the baseline decisively** on probabilistic metrics — the gap is much
  wider there than on raw accuracy, which is exactly what you would expect. The models are not
  guessing outcomes better so much as expressing uncertainty better.
- **The estimated home advantage is stable** across model specifications, matching the
  exploratory finding.

### Honest limitations

1. **No squad information.** Injuries, suspensions and transfers are invisible to the model.
   A team missing its top scorer carries the same rating as a fully fit one.
2. **Promoted teams are guesswork.** Newly promoted sides receive replacement-level
   parameters until they accumulate matches — their early-season forecasts are the least
   reliable in the set.
3. **No European or cup fixture congestion.** Teams playing midweek in Europe are
   systematically disadvantaged, and the model does not know this.
4. **Accuracy has a hard ceiling.** Football is genuinely high-variance. Bookmakers with
   substantial resources operate in a similar accuracy range; the value of a model lies in
   calibration, not in beating a coin flip by a wide margin.

### Next step

Forecasts are written to `data/predictions/upcoming_forecasts.csv` on every run. As the season
progresses, these can be scored against actual results — turning the project from a
retrospective analysis into a live, verifiable track record.